# Forward propagation
One row through the 4-3-2-1 network, with the hand-set weights of the Note; then the same weights in scikit-learn and Keras.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# one student: CGPA 7.2, IQ 72, 10th 69, 12th 81, scaled into 0..1
raw = np.array([7.2, 72, 69, 81])
a0 = raw / np.array([10, 100, 100, 100])
print(a0)

In [ ]:
# weights: shape (nodes in previous layer, nodes in this layer), so W1[i-1, j-1] is W^1_ij
W1 = np.array([[0.2, -0.3, 0.5], [0.4, 0.1, -0.2], [-0.5, 0.2, 0.1], [0.3, -0.4, 0.2]])
b1 = np.array([0.1, -0.1, 0.2])
W2 = np.array([[0.6, -0.4], [-0.2, 0.5], [0.3, 0.7]])
b2 = np.array([0.1, -0.2])
W3 = np.array([[0.8], [-0.6]])
b3 = np.array([0.2])

# trainable parameters: every weight and every bias
print(sum(p.size for p in [W1, b1, W2, b2, W3, b3]))   # 26

## Node 1 of layer 1 by hand

In [ ]:
z = 0.2 * 0.72 + 0.4 * 0.72 - 0.5 * 0.69 + 0.3 * 0.81 + 0.1
print(round(z, 3), round(sigmoid(z), 3))     # 0.43, 0.606

## Layer by layer

In [ ]:
z1 = W1.T @ a0 + b1      # (3x4)(4x1) + (3x1)
a1 = sigmoid(z1)
z2 = W2.T @ a1 + b2      # (2x3)(3x1) + (2x1)
a2 = sigmoid(z2)
z3 = W3.T @ a2 + b3      # (1x2)(2x1) + (1x1)
a3 = sigmoid(z3)
for name, v in [("W1.T @ a0", W1.T @ a0), ("z1", z1), ("a1", a1), ("z2", z2), ("a2", a2), ("z3", z3), ("y_hat", a3)]:
    print(f"{name:10s}", v.round(3))

## The nested formula and the loop give the same answer

In [ ]:
nested = sigmoid(W3.T @ sigmoid(W2.T @ sigmoid(W1.T @ a0 + b1) + b2) + b3)

a = a0
for W, b in [(W1, b1), (W2, b2), (W3, b3)]:
    a = sigmoid(W.T @ a + b)
print(nested, a)

## Why we scale: raw inputs push every sigmoid to an extreme

In [ ]:
z_raw = W1.T @ raw + b1
print(z_raw.round(2), sigmoid(z_raw).round(6))

## Many rows at once: X @ W, no transpose

In [ ]:
rng = np.random.default_rng(0)
X = np.vstack([a0, rng.uniform(0.4, 1.0, size=(4, 4))])    # 5 students, row 0 is ours
A = X
for W, b in [(W1, b1), (W2, b2), (W3, b3)]:
    A = sigmoid(A @ W + b)            # (n x in)(in x out) + bias
print(A.ravel().round(3))             # first value 0.594

## Check with scikit-learn's MLPClassifier
We fit it briefly only to create the arrays, then overwrite them with our weights.

In [ ]:
import warnings
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
clf = MLPClassifier(hidden_layer_sizes=(3, 2), activation="logistic", max_iter=1, random_state=0)
clf.fit(X, [1, 0, 1, 0, 1])           # any labels: we only need the model built
print([w.shape for w in clf.coefs_], [b.shape for b in clf.intercepts_])
clf.coefs_ = [W1, W2, W3]
clf.intercepts_ = [b1, b2, b3]
print(clf.predict_proba(a0.reshape(1, -1))[0, 1])     # 0.594

## Check with Keras
The same network as a Keras `Sequential` model: three `Dense` layers with sigmoid activation. `set_weights` takes the kernel (shape: nodes in, nodes out) and the bias of each layer.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow's start-up messages
import keras

keras.utils.set_random_seed(0)
model = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(3, activation="sigmoid"),
    keras.layers.Dense(2, activation="sigmoid"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.summary()                               # 15 + 8 + 3 = 26 parameters
model.set_weights([W1, b1, W2, b2, W3, b3])
print(model.predict(a0.reshape(1, -1), verbose=0))   # [[0.594]]